In [24]:
import os
from openai import AzureOpenAI
import openai
from openai import OpenAI
import pandas as pd
import spacy
import scispacy
import pandas as pd
import math
import numpy as np
import requests
import re
from collections import Counter
from sklearn.preprocessing import MultiLabelBinarizer
import re
from typing import Dict, List, Optional, Set, Tuple, Any

In [5]:
import os
from pathlib import Path

# Define the correct folder path
target_folder = Path("/Users/naveendeevi/Python Projects/MDD/MDD_Datasets")

# Check if folder exists
if target_folder.exists() and target_folder.is_dir():
    os.chdir(target_folder)
    print(f"Current working directory: {os.getcwd()}")

    # List all items in the folder
    items = list(target_folder.iterdir())
    if items:
        print("\nItems in directory:")
        for item in items:
            print(f"- {item.name}")
    else:
        print("\nNo items found in the directory.")
else:
    raise FileNotFoundError(f"Folder not found: {target_folder}")

Current working directory: /Users/naveendeevi/Python Projects/MDD/MDD_Datasets

Items in directory:
- openfda_cache_dosage.json
- All_Medications_MDD.numbers
- MDD_Drugs_V1_1.numbers
- Similarity Drugs_V2.numbers
- MDD_Drugs_V1.numbers
- Similarity Drugs_V2.csv
- test.csv
- drug_similarity 2.numbers
- Indications_to_sertraline.csv
- drug_test.csv
- mechanisms_and_targets 2.numbers
- drug_test_copy.csv
- mechanisms_and_targets.numbers
- mechanisms_and_targets 2.csv
- All_Medications_MDD.csv
- MDD Pat Count with Subtypes_V1.csv
- drug_similarity.numbers
- MOA\Target Validation.numbers
- .openfda_pk_cache
- .ipynb_checkpoints
- score_tfidf_symantic V3 2.numbers
- score_tfidf_symantic V2.numbers
- MDD_Drugs_V1.csv
- MDD Subtypes_V1.numbers
- mechanisms_and_targets.csv
- test.numbers
- score_tfidf_symantic V3.numbers
- df_pharmacokinetics.csv
- openfda_cache.json
- openfda_cache_clinical.json


In [25]:
# ==================== Tokenizes the target field ====================

# ==================== Helpers ====================

# ==================== Config ====================

# Functional / descriptor words to strip anywhere in the phrase.
# (Additions: 'like', 'voltage', 'gated', 'ligand', plus 'complex', 'group')
_STOPWORDS = {
    "receptor", "receptors", "transporter", "transporters",
    "channel", "channels", "enzyme", "enzymes", "subunit", "subunits",
    "carrier", "carriers", "protein", "proteins",
    "complex", "group", "like", "voltage", "gated", "ligand"
}

# Canonical families. Use desired casing per family.
# Note: ions kept lowercase per your examples; neurotransmitters title/upper-cased.
_CANONICAL_FAMILY = {
    "serotonin": "Serotonin",
    "dopamine": "Dopamine",
    "norepinephrine": "Norepinephrine",
    "adrenergic": "Adrenergic",
    "histamine": "Histamine",
    "cholinergic": "Cholinergic",
    "gaba": "GABA",
    "glutamate": "Glutamate",
    "opioid": "Opioid",
    "cannabinoid": "Cannabinoid",
    # ions / generic classes (lowercase as requested behavior)
    "calcium": "calcium",
    "sodium": "sodium",
    "potassium": "potassium",
    "chloride": "chloride",
    "anion": "anion",
    "cation": "cation",
}


# Generalized prefix stripper: handles
# SINGLE PROTEIN:, PROTEIN FAMILY:, PROTEIN COMPLEX:, PROTEIN COMPLEX GROUP:, etc.
_TYPE_PREFIX_RE = re.compile(
    r"^\s*(?:[A-Z ]*PROTEIN(?:\s+[A-Z-]+)*)\s*:\s*",
    flags=re.IGNORECASE
)

# Greek/letter class mapping for adrenergic/GABA-like subtypes
_GREEK_MAP = {
    "alpha": "A", "a": "A", "α": "A",
    "beta": "B",  "b": "B", "β": "B",
    "gamma": "G", "g": "G", "γ": "G",
    "delta": "D", "d": "D", "δ": "D",
}

def _strip_type_prefix(chunk: str) -> str:
    return _TYPE_PREFIX_RE.sub("", chunk, count=1)

def _split_top_level(field: str) -> List[str]:
    """Stage A: split on ';;;' (treat newlines as ';;;')."""
    s = field.replace("\n", ";;;")
    return [c.strip() for c in re.split(r";;;+", s) if c.strip()]

def _split_semicolons_outside(s: str) -> List[str]:
    """
    Stage B: split on single ';' outside () and []
    e.g., 'GABA-A receptor; anion channel' -> ['GABA-A receptor', 'anion channel']
    """
    parts, buf = [], []
    paren, bracket = 0, 0
    for ch in s:
        if ch == '(':
            paren += 1
        elif ch == ')':
            paren = max(0, paren - 1)
        elif ch == '[':
            bracket += 1
        elif ch == ']':
            bracket = max(0, bracket - 1)

        if ch == ';' and paren == 0 and bracket == 0:
            part = ''.join(buf).strip()
            if part:
                parts.append(part)
            buf = []
        else:
            buf.append(ch)
    last = ''.join(buf).strip()
    if last:
        parts.append(last)
    return parts

def _drop_paren_and_brackets(s: str) -> str:
    """Remove (...) and [...] contents."""
    s = re.sub(r"\([^)]*\)", " ", s)
    s = re.sub(r"\[[^\]]*\]", " ", s)
    return s

def _normalize_leading_synonyms(s: str) -> str:
    """
    Normalize leading synonyms and punctuation; return lowercased, space-normalized string.
    """
    s = s.strip()
    # Normalize separators to spaces (keep ';'—already split)
    s = re.sub(r"[-_/]", " ", s)
    # Remove trailing punctuation we don't need (keep digits/letters)
    s = re.sub(r"[.,:]+", " ", s)
    s = re.sub(r"\s+", " ", s)
    s_low = s.lower()

    # 5-HT -> serotonin
    s_low = re.sub(r"^\s*5\s*ht\b", "serotonin", s_low)

    # -ergic adjectives to base families
    s_low = re.sub(r"^\s*serotonin(ergic)?\b", "serotonin", s_low)
    s_low = re.sub(r"^\s*dopamin(ergic)?\b", "dopamine", s_low)
    s_low = re.sub(r"^\s*histamin(ergic)?\b", "histamine", s_low)
    s_low = re.sub(r"^\s*cholin(ergic)?\b", "cholinergic", s_low)
    s_low = re.sub(r"^\s*gaba(ergic)?\b", "gaba", s_low)
    s_low = re.sub(r"^\s*glutamat(ergic|e)?\b", "glutamate", s_low)
    s_low = re.sub(r"^\s*adrenergic\b", "adrenergic", s_low)

    # Noradrenaline → norepinephrine
    s_low = re.sub(r"^\s*noradren(alin(e)?|ergic)\b", "norepinephrine", s_low)

    return s_low

def _tokenize_clean_phrase(raw_phrase: str) -> List[str]:
    """
    Remove parentheses/brackets, normalize punctuation/spacing, lowercase, split,
    and drop stopwords.
    """
    s = _drop_paren_and_brackets(raw_phrase)
    s = _normalize_leading_synonyms(s)
    tokens = [t for t in re.split(r"\s+", s.strip()) if t]
    tokens = [t for t in tokens if t not in _STOPWORDS]
    return tokens

def _normalize_simple_subtype(tok: str) -> Optional[str]:
    """
    Normalize single-token subtype candidates:
      '1a' -> '1A', 'd2' -> 'D2', 'alpha2'/'β1'/'a2' -> 'A2', etc.
    Return None if not a subtype.
    """
    t = tok.strip().lower()

    # Compact greek+digit forms (alpha2, β1, a2)
    m = re.match(r"^(alpha|beta|gamma|delta|a|b|g|d|α|β|γ|δ)\s*(\d+)$", t)
    if m:
        return f"{_GREEK_MAP[m.group(1)]}{m.group(2)}"

    # Digit + letter (e.g., 1a)
    m = re.match(r"^(\d+)\s*([a-z])$", t)
    if m:
        return f"{m.group(1)}{m.group(2).upper()}"

    # Letter + digit (e.g., d2)
    m = re.match(r"^([a-z])\s*(\d+)$", t)
    if m:
        return f"{m.group(1).upper()}{m.group(2)}"

    # Pure letter class (A/B/G/D) sometimes used after GABA (e.g., 'gaba a')
    if len(t) == 1 and t in _GREEK_MAP:
        return _GREEK_MAP[t]

    return None

def _select_family(tokens: List[str]) -> (Optional[str], Optional[int]):
    """
    Choose family token with priority:
      1) First token that matches a known family (any position)
      2) Fallback to the first token.
    Returns (family_string, index_in_tokens) or (None, None).
    """
    if not tokens:
        return None, None

    known = set(_CANONICAL_FAMILY.keys())
    for i, tok in enumerate(tokens):
        if tok in known:
            fam = _CANONICAL_FAMILY.get(tok, tok.capitalize())
            return fam, i

    # Fallback: first token, capitalized unless we have a canonical override
    first = tokens[0]
    fam = _CANONICAL_FAMILY.get(first, first.capitalize())
    return fam, 0

def _find_subtype(tokens: List[str], family_idx: Optional[int]) -> Optional[str]:
    """
    Find subtype by scanning tokens around the family token.
    Handles 'D2-like dopamine receptor' (subtype appears before family).
    """
    if not tokens:
        return None

    # Build candidate indices to scan: tokens other than the family token,
    # first scan to the right, then to the left to bias towards patterns near family.
    indices = list(range(len(tokens)))
    if family_idx is not None and 0 <= family_idx < len(tokens):
        indices = [i for i in range(family_idx + 1, len(tokens))] + [i for i in range(family_idx) ]

    # Pairwise patterns (alpha 2) next to each other
    for i in indices:
        t = tokens[i]

        # Pair (alpha 2) -> 'A2'
        if t in _GREEK_MAP and i + 1 < len(tokens) and tokens[i + 1].isdigit():
            return f"{_GREEK_MAP[t]}{tokens[i + 1]}"

        # Merge with neighbor if needed (handles 'alpha' + '-' + '2' cases after normalization)
        if i + 1 < len(tokens):
            merged = t + tokens[i + 1]
            cand2 = _normalize_simple_subtype(merged)
            if cand2:
                return cand2

        # Single-token subtype like '1a', 'd2', 'beta1'
        cand = _normalize_simple_subtype(t)
        if cand:
            return cand

    return None

def _extract_family_and_subtype(tokens: List[str]) -> Optional[Dict[str, Optional[str]]]:
    """
    tokens: cleaned, lowercased tokens with stopwords removed.
    Returns dict: {'family': str, 'subtype': Optional[str], 'family_subtype': Optional[str]}
    or None if no family could be parsed.
    """
    if not tokens:
        return None

    family, fam_idx = _select_family(tokens)
    if not family or not re.search(r"[A-Za-z]", family):
        return None

    subtype = _find_subtype(tokens, fam_idx)
    family_subtype = f"{family} {subtype}" if subtype else None
    return {"family": family, "subtype": subtype, "family_subtype": family_subtype}

def _dedup_preserve_order(items: List[Optional[str]]) -> List[str]:
    seen = set()
    out: List[str] = []
    for x in items:
        if x is None:
            continue
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def tokenize_targets_field(field: Optional[str]) -> Dict[str, List[Optional[str]]]:
    """
    Parse multi-target fields like:
      'SINGLE PROTEIN:Serotonin 2a (5-HT2a) receptor;;;PROTEIN FAMILY:D2-like dopamine receptor'
      'PROTEIN COMPLEX GROUP:Voltage-gated calcium channel'
      'PROTEIN COMPLEX GROUP:Glutamate [NMDA] receptor;;;PROTEIN COMPLEX GROUP:GABA-A receptor; anion channel'

    Returns:
      {
        "entries": [ {family, subtype, family_subtype}, ... ],
        "families": [unique families, order-preserving],
        "subtypes": [subtype per entry (may include None)],
        "family_subtypes": [unique family_subtype labels (skips None), order-preserving]
      }
    """
    if not field or not isinstance(field, str):
        return {"entries": [], "families": [], "subtypes": [], "family_subtypes": []}

    entries: List[Dict[str, Optional[str]]] = []
    families_all: List[Optional[str]] = []
    subtypes_all: List[Optional[str]] = []
    family_subtypes_all: List[Optional[str]] = []

    # Stage A: top-level ';;;' split
    top_chunks = _split_top_level(field)

    for chunk in top_chunks:
        phrase_after_prefix = _strip_type_prefix(chunk)

        # Stage B: split intra-chunk on ';' outside () and []
        for sub in _split_semicolons_outside(phrase_after_prefix):
            tokens = _tokenize_clean_phrase(sub)
            parsed = _extract_family_and_subtype(tokens)
            if not parsed:
                continue

            entries.append(parsed)
            families_all.append(parsed["family"])
            subtypes_all.append(parsed["subtype"])
            family_subtypes_all.append(parsed["family_subtype"])

    return {
        "entries": entries,
        "families": _dedup_preserve_order(families_all),
        "subtypes": subtypes_all,
        "family_subtypes": _dedup_preserve_order(family_subtypes_all),
    }

#print(tokenize_targets_field(e))


## Function to create the fields of 'families' & 'subtypes' 

def _dedup_preserve_order_list(seq):
    """Return a list with duplicates removed, preserving first-seen order."""
    seen = set()
    out = []
    for x in seq or []:
        if x is None or x == "":
            continue
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


# Reuse the helpers from the previous snippet -------------------------------

def _as_set_listlike(val) -> Set[str]:
    """Safely convert a list-like cell to a set of str, dropping None/empty."""
    if isinstance(val, (list, tuple, set)):
        return {str(x) for x in val if x is not None and str(x) != ""}
    # handle NaN / non-listlike
    return set()

def _jaccard(a: Set[str], b: Set[str]) -> float:
    """Standard Jaccard similarity (0 if both empty)."""
    if not a and not b:
        return 0.0
    inter = len(a & b)
    union = len(a | b)
    return inter / union if union > 0 else 0.0

def _pair_similarity(
    Fi: Set[str], Fj: Set[str], Ti: Set[str], Tj: Set[str]
) -> Tuple[float, Optional[float], float, Set[str]]:
    """
    Step 1: families Jaccard
    Step 2: subtypes Jaccard (ONLY if Step 1 > 0; and ONLY if the INDEX drug has subtypes)
    Combine: 50/50 if Step 2 computed; else 100% Step 1
    Returns: (s1, s2, combined, shared_families)
    """
    shared_families = Fi & Fj
    s1 = _jaccard(Fi, Fj)

    # If no family overlap (s1 == 0), stop here.
    if s1 <= 0.0:
        return 0.0, None, 0.0, shared_families

    # If index drug has no subtypes, skip Step 2 and give 100% weight to families.
    if not Ti:
        return s1, None, s1, shared_families

    # Otherwise compute subtype Jaccard on overall subtype sets.
    s2 = _jaccard(Ti, Tj)
    combined = 0.5 * s1 + 0.5 * s2
    return s1, s2, combined, shared_families

# New: compute similarity TO A REFERENCE drug --------------------------------

def compute_similarity_to_reference(
    df: pd.DataFrame,
    *,
    ref_id: Optional[str] = None,
    ref_idx: Optional[int] = None,
    id_col: str = "Drug",
    families_col: str = "families",
    subtypes_col: str = "subtypes",
) -> pd.DataFrame:
    """
    Compute similarity from a single *reference* drug to all others (directed),
    following your rules:
      - Step 1: Jaccard on families
      - Step 2: Jaccard on subtypes (only if Step 1 > 0; and only if reference has subtypes)
      - Combined: 50/50 if Step 2 is computed; else 100% Step 1

    Choose the reference either by `ref_id` (matching df[id_col]) OR by `ref_idx`
    (0-based row index). One of them must be provided.

    Returns a DataFrame with one row per OTHER drug:
      ['ref_id','other_id','families_ref','families_other','subtypes_ref','subtypes_other',
       'shared_families','step1_families_jaccard','step2_subtypes_jaccard','combined_score']
    """
    if ref_id is None and ref_idx is None:
        raise ValueError("Provide either ref_id or ref_idx.")

    # Locate reference row
    if ref_idx is not None:
        if ref_idx < 0 or ref_idx >= len(df):
            raise IndexError(f"ref_idx out of bounds: {ref_idx}")
        ref_row = df.iloc[ref_idx]
    else:
        matches = df[df[id_col] == ref_id]
        if matches.empty:
            raise ValueError(f"ref_id '{ref_id}' not found in column '{id_col}'.")
        ref_row = matches.iloc[0]

    ref_id_val = ref_row[id_col]
    Fi = _as_set_listlike(ref_row.get(families_col, []))
    Ti = _as_set_listlike(ref_row.get(subtypes_col, []))

    out_rows: List[Dict[str, Any]] = []
    for _, r in df.iterrows():
        other_id = r[id_col]
        if other_id == ref_id_val:
            continue  # skip self

        Fj = _as_set_listlike(r.get(families_col, []))
        Tj = _as_set_listlike(r.get(subtypes_col, []))

        s1, s2, comb, shared = _pair_similarity(Fi, Fj, Ti, Tj)

        out_rows.append({
            "ref_id": ref_id_val,
            "other_id": other_id,
            "families_ref": sorted(Fi),
            "families_other": sorted(Fj),
            "subtypes_ref": sorted(Ti),
            "subtypes_other": sorted(Tj),
            "shared_families": sorted(shared),
            "step1_families_jaccard": s1,
            "step2_subtypes_jaccard": s2,   # NaN when Step 2 skipped or Step 1 == 0
            "combined_score": comb,
        })

    return pd.DataFrame(out_rows).sort_values("combined_score", ascending=False).reset_index(drop=True)




In [15]:
# Main function
df_compare = pd.read_csv('MDD_Drugs_V1.csv')
df_compare_2 = df_compare[['ingredient_name']]
df_compare_3 = df_compare_2.drop_duplicates()

df = pd.read_csv('mechanisms_and_targets 2.csv')
df_2 = df[['DrugName','MOA','Targets']]

df_3 = pd.merge(
    df_compare_3,
    df_2,
    how="left",             # inner join
    left_on="ingredient_name",      # column in df_3
    right_on="DrugName"  # column in df_compare_3
)

df_4 = df_3[['ingredient_name','MOA','Targets']]

In [53]:
df_4.head(25)

,ingredient_name,MOA,Targets,tokenized,families,subtypes
0,Amphetamine / Dextroamphetamine,NaN,NaN,"{'entries': [], 'families': [], 'subtypes': []...",[],[]
1,aripiprazole,PARTIAL AGONIST:Dopamine D2 receptor partial a...,SINGLE PROTEIN:Dopamine D2 receptor;;;SINGLE P...,"{'entries': [{'family': 'Dopamine', 'subtype':...","[Dopamine, Serotonin]","[D2, 1A, 2A]"
2,clonidine hydrochloride,AGONIST:Adrenergic receptor alpha-2 agonist,PROTEIN FAMILY:Adrenergic receptor alpha-2,"{'entries': [{'family': 'Adrenergic', 'subtype...",[Adrenergic],[A2]
3,NaN,NaN,NaN,"{'entries': [], 'families': [], 'subtypes': []...",[],[]
4,asenapine,NaN,NaN,"{'entries': [], 'families': [], 'subtypes': []...",[],[]
5,bupropion hydrochloride,INHIBITOR:Dopamine transporter inhibitor;;;INH...,SINGLE PROTEIN:Dopamine transporter;;;SINGLE P...,"{'entries': [{'family': 'Dopamine', 'subtype':...","[Dopamine, Norepinephrine]",[]
6,desvenlafaxine succinate,NaN,NaN,"{'entries': [], 'families': [], 'subtypes': []...",[],[]
7,divalproex sodium,INHIBITOR:Succinate semialdehyde dehydrogenase...,SINGLE PROTEIN:Succinate semialdehyde dehydrog...,"{'entries': [{'family': 'Succinate', 'subtype'...",[Succinate],[]
8,lamotrigine,BLOCKER:Sodium channel alpha subunit blocker,PROTEIN FAMILY:Sodium channel alpha subunit,"{'entries': [{'family': 'sodium', 'subtype': N...",[sodium],[]
9,methylphenidate hydrochloride,INHIBITOR:Dopamine transporter inhibitor;;;INH...,SINGLE PROTEIN:Dopamine transporter;;;SINGLE P...,"{'entries': [{'family': 'Dopamine', 'subtype':...","[Dopamine, Norepinephrine]",[]


In [60]:
# --- apply to your dataframe ---
# assume your column is named 'targets' (a list of strings per row)
# replace 'targets' with your real column name
#df_4["targets_parsed"] = df_4["target_tokenized"].apply(parse_targets_combined)

df_4["tokenized"] = df_4["Targets"].apply(tokenize_targets_field)

# 1) families: already unique/order-preserving from tokenize_targets_field,
#    but we defensively de-dup just in case.
df_4["families"] = df_4["tokenized"].apply(
    lambda d: _dedup_preserve_order_list(d.get("families", [])) if isinstance(d, dict) else []
)

# 2) subtypes: the tokenizer returns per-entry subtypes (may include None/dups),
#    so we drop None/empty and de-dup while preserving order.
df_4["subtypes"] = df_4["tokenized"].apply(
    lambda d: _dedup_preserve_order_list(d.get("subtypes", [])) if isinstance(d, dict) else []
)


# -------------------- Example usage --------------------
# 1) To compute similarity from a single reference by ID:
sim_to_ref = compute_similarity_to_reference(df_4, ref_id="armodafinil",id_col="ingredient_name",families_col="families",subtypes_col="subtypes") 
# sim_to_ref.head(10)

# 2) Or pick the reference by row index:
# sim_to_ref = compute_similarity_to_reference(df_4,
#                                              ref_idx=0,
#                                              id_col="Drug",
#                                              families_col="families",
#                                              subtypes_col="subtypes")

# 3) To build a full *directed* table for every i -> j:
# sim_directed = compute_all_pairs_directed(df_4,
#                                           id_col="Drug",
#                                           families_col="families",
#                                           subtypes_col="subtypes")
# sim_directed.head()


In [61]:
#df_4.head(20)

In [62]:
sim_to_ref.loc[sim_to_ref['combined_score'] > 0.0]

,ref_id,other_id,families_ref,families_other,subtypes_ref,subtypes_other,shared_families,step1_families_jaccard,step2_subtypes_jaccard,combined_score
0,armodafinil,fluphenazine hydrochloride,[Dopamine],[Dopamine],[],[D2],[Dopamine],1.000000,None,1.000000
1,armodafinil,cariprazine hydrochloride,[Dopamine],[Dopamine],[],"[D2, D3]",[Dopamine],1.000000,None,1.000000
2,armodafinil,modafinil,[Dopamine],[Dopamine],[],[],[Dopamine],1.000000,None,1.000000
3,armodafinil,perphenazine,[Dopamine],[Dopamine],[],[D2],[Dopamine],1.000000,None,1.000000
4,armodafinil,quetiapine,[Dopamine],"[Dopamine, Serotonin]",[],"[2A, 2C, D2]",[Dopamine],0.500000,None,0.500000
5,armodafinil,quetiapine fumarate,[Dopamine],"[Dopamine, Serotonin]",[],"[2A, 2C, D2]",[Dopamine],0.500000,None,0.500000
6,armodafinil,aripiprazole,[Dopamine],"[Dopamine, Serotonin]",[],"[1A, 2A, D2]",[Dopamine],0.500000,None,0.500000
7,armodafinil,dexmethylphenidate hydrochloride,[Dopamine],"[Dopamine, Norepinephrine]",[],[],[Dopamine],0.500000,None,0.500000
8,armodafinil,iloperidone,[Dopamine],"[Dopamine, Serotonin]",[],"[2A, D2]",[Dopamine],0.500000,None,0.500000
9,armodafinil,lurasidone hydrochloride,[Dopamine],"[Dopamine, Serotonin]",[],"[2A, D2]",[Dopamine],0.500000,None,0.500000
